# Fase 2 · ferramentas
O ReAct clássico fica isolado. O AutoGen organiza conversas entre agentes; aqui o integrador usa um agente com ferramentas verificáveis.

In [1]:
from pathlib import Path
import os
if Path.cwd().name == 'notebooks': os.chdir('..')

In [2]:
from pedro.catalog import criar_cenario
from pedro.legacy import RouterCompras, executar_rota
from pedro.chains import composicao_paralela
from pedro.integrations import consultar_cep
cenario = criar_cenario(Path('storage/notebooks/catalogos'))
router = RouterCompras()
print(executar_rota(router, cenario, 'preço de PEC-01-0001'))
print(executar_rota(router, cenario, 'PEC-01-0001'))
print(composicao_paralela(cenario).invoke({'produto':'Compressor'}))
print(consultar_cep('01001000', simulado=True))

{'rota': 'comparacao', 'resultado': [{'dataset_id': '4da8a8c4-ceb0-44a3-b314-bc412b249a92', 'id': 'PEC-01-0001-BR-05', 'produto_id': 'PEC-01-0001', 'fornecedor_id': 'BR-05', 'fornecedor_nome': 'Litoral Parts', 'preco_unitario_centavos': 37020, 'frete_centavos': 7940, 'estoque': 0, 'prazo_dias': 12, 'pedido_minimo': 5, 'nacional': True, 'meses_historico': 5, 'preco_anterior_centavos': 42977, 'preco_medio_centavos': 41786, 'preco_minimo_centavos': 37020, 'preco_maximo_centavos': 45956}, {'dataset_id': '4da8a8c4-ceb0-44a3-b314-bc412b249a92', 'id': 'PEC-01-0001-BR-06', 'produto_id': 'PEC-01-0001', 'fornecedor_id': 'BR-06', 'fornecedor_nome': 'CentroPeças', 'preco_unitario_centavos': 143892, 'frete_centavos': 8523, 'estoque': 67, 'prazo_dias': 2, 'pedido_minimo': 5, 'nacional': True, 'meses_historico': 5, 'preco_anterior_centavos': 118987, 'preco_medio_centavos': 137251, 'preco_minimo_centavos': 118987, 'preco_maximo_centavos': 159111}, {'dataset_id': '4da8a8c4-ceb0-44a3-b314-bc412b249a92',

## Roteamento dinâmico e composição
A primeira célula deste notebook executa `RouterCompras` (uma `RouterChain` com regras), `executar_rota` (cadeia sequencial: decidir a rota e executar a cadeia dela) e `composicao_paralela` (`RunnableParallel`). Aqui se acrescenta o router decidido por um modelo (`LLMRouterChain`). **Duplo de teste:** `FakeListLLM` devolve o destino já escrito; a comparação com o modelo real está em `docs/rastros/r09_router_llm.json`.

In [3]:
from langchain_core.language_models.fake import FakeListLLM
from pedro.legacy import DESTINOS, criar_router_llm
print('destinos:', DESTINOS)
for pergunta, destino in (('Quanto custa o compressor PEC-01-0001?', 'comparacao'),
                          ('A qual cidade corresponde o CEP 01001-000?', 'cep'),
                          ('Olá, bom dia', 'DEFAULT')):
    duplo = FakeListLLM(responses=['```json\n{"destination": "' + destino + '", "next_inputs": "x"}\n```'])
    saida = executar_rota(criar_router_llm(duplo), cenario, pergunta)
    print(pergunta, '->', saida['rota'], '|', str(saida['resultado'])[:110])
    print('   o router por regras teria escolhido:', executar_rota(router, cenario, pergunta)['rota'])

destinos: {'catalogo': 'buscar ou descrever uma peça por nome, SKU ou ID, sem falar de preços', 'comparacao': 'preços, custos, ofertas, fornecedores, estoque ou prazos de uma peça', 'cep': 'confirmar a cidade ou o estado de um código postal brasileiro (CEP)'}
Quanto custa o compressor PEC-01-0001? -> comparacao | [{'dataset_id': '4da8a8c4-ceb0-44a3-b314-bc412b249a92', 'id': 'PEC-01-0001-BR-05', 'produto_id': 'PEC-01-0001'
   o router por regras teria escolhido: catalogo
A qual cidade corresponde o CEP 01001-000? -> cep | {'estado': 'ok', 'cep': '01001000', 'cidade': 'Cidade Simulada', 'uf': 'SP', 'modo': 'simulado'}
   o router por regras teria escolhido: catalogo
Olá, bom dia -> esclarecimento | Você procura uma peça, as ofertas dela ou um CEP?
   o router por regras teria escolhido: catalogo


## Agente ReAct com initialize_agent, ferramentas e memória
**Duplo de teste:** `FakeListLLM` devolve um roteiro fixo de pensamentos e ações; não decide nada. O real é o agente (`initialize_agent` com `ZERO_SHOT_REACT_DESCRIPTION`), as duas ferramentas executadas sobre o catálogo e `ConversationBufferMemory`. A mesma conversa com o modelo real, em que o LLM de fato escolhe, está em `docs/rastros/r08_react_llm.json`.

In [4]:
from langchain_core.language_models.fake import FakeListLLM
from pedro.legacy import criar_react_textual
llm = FakeListLLM(responses=[
    'Thought: devo buscar a peça\nAction: buscar_peca\nAction Input: GELA-COMP-0001',
    'Thought: já a tenho\nFinal Answer: o ID é PEC-01-0001',
    'Thought: o histórico diz PEC-01-0001\nAction: ofertas_da_peca\nAction Input: PEC-01-0001',
    'Thought: comparo os prazos\nFinal Answer: CentroPeças entrega em 2 dias'])
react, buffer = criar_react_textual(llm, cenario)
print('variáveis do prompt:', sorted(react.agent.llm_chain.prompt.input_variables))
for pergunta in ('Busque o compressor com SKU GELA-COMP-0001 e me diga o ID dele.',
                 'Qual dos fornecedores dele entrega mais rápido?'):
    saida = react.invoke({'input': pergunta})
    for acao, observacao in saida['intermediate_steps']:
        print('  ação:', acao.tool, '| entrada:', acao.tool_input, '| observação:', str(observacao)[:150])
    print('  resposta:', saida['output'])
print('memória de buffer:')
print(buffer.load_memory_variables({})['chat_history'])

variáveis do prompt: ['agent_scratchpad', 'chat_history', 'input']
  ação: buscar_peca | entrada: GELA-COMP-0001 | observação: [{'id': 'PEC-01-0001', 'sku': 'GELA-COMP-0001', 'name': 'Compressor para Geladeiras GELA-COMP-0001', 'category': 'Geladeiras'}]
  resposta: o ID é PEC-01-0001
  ação: ofertas_da_peca | entrada: PEC-01-0001 | observação: [{'fornecedor': 'Litoral Parts', 'preco_unitario_centavos': 37020, 'estoque': 0, 'prazo_dias': 12}, {'fornecedor': 'CentroPeças', 'preco_unitario_cent
  resposta: CentroPeças entrega em 2 dias
memória de buffer:
Human: Busque o compressor com SKU GELA-COMP-0001 e me diga o ID dele.
AI: o ID é PEC-01-0001
Human: Qual dos fornecedores dele entrega mais rápido?
AI: CentroPeças entrega em 2 dias


/home/efraxpc/infnet/agentes/pedro_asistente_de_compras/src/pedro/legacy.py:78: LangChainDeprecationWarning: The class `ConversationBufferMemory` was deprecated in LangChain 0.3.1 and will be removed in 2.0.0. Use `langchain.agents.create_agent` instead. For agents that need to remember prior interactions, use `create_agent` with checkpointing or the `Store` API. See https://docs.langchain.com/oss/python/langchain/short-term-memory and https://docs.langchain.com/oss/python/langchain/long-term-memory
  memoria = ConversationBufferMemory(memory_key="chat_history", input_key="input", output_key="output")
/home/efraxpc/infnet/agentes/pedro_asistente_de_compras/.venv/lib/python3.13/site-packages/pydantic/main.py:280: LangChainDeprecationWarning: The class `InMemoryChatMessageHistory` was deprecated in LangChain 1.6.4 and will be removed in 2.0.0 See the short-term memory documentation for recommended alternatives: https://docs.langchain.com/oss/python/langchain/short-term-memory
  validated

A chamada real ao ViaCEP está registrada separadamente em docs/rastros/r10_viacep_llm.json.

## Registro de um turno com erro e recuperação
Os quatro componentes de confiabilidade em um único turno do fluxo final. **Duplo de teste:** o modelo é um roteiro; o ViaCEP está simulado. A primeira proposta do roteiro é texto que não é JSON (erro do parser); a segunda é válida. O callback `Rastro` registra cada chamada de ferramenta e cada verificação, e `RunnableWithMessageHistory` salva na memória só a pergunta e a resposta publicada.

In [5]:
import json
from langchain_core.messages import AIMessage
from langchain_core.runnables import RunnableLambda
from pedro.agent import Rastro, criar_fluxo
from pedro.memory import Historico, nova_sessao
memoria = Path('storage/notebooks/registro_do_turno.sqlite3')
sessao = nova_sessao(memoria, 'aluna', cenario.dataset_id)
chamar = AIMessage(content='', tool_calls=[{'name': 'consultar_cep', 'args': {'cep': '01001000'}, 'id': 'c1'}])
valida = json.dumps({'estado': 'cep', 'dataset_id': cenario.dataset_id, 'solicitacao': {'cep': '01001000'}})
roteiro = iter([chamar, AIMessage(content='A cidade é São Paulo, acho.'), chamar, AIMessage(content=valida)])
rastro = Rastro()
fluxo = criar_fluxo(cenario, memoria, llm=RunnableLambda(lambda prompt, **kw: next(roteiro)),
                    cep_simulado=True, rastro=rastro)
resposta = fluxo.invoke({'input': 'A qual cidade corresponde o CEP 01001000?'},
                        config={'configurable': {'user_id': 'aluna', 'session_id': sessao}})
for evento in rastro.eventos:
    print({k: (str(v)[:70] if isinstance(v, str) else v) for k, v in evento.items()})
print('publicado:', resposta.content)
print('memória:', [(m.type, m.content[:60]) for m in Historico(memoria, 'aluna', sessao).todos()])

{'evento': 'tool_start', 'rota': 'consultar_cep', 'ferramenta': 'consultar_cep', 'entrada': "{'cep': '01001000'}"}
{'evento': 'tool_end', 'resultado': '{"estado": "ok", "cep": "01001000", "cidade": "Cidade Simulada", "uf":', 'fontes': []}
{'evento': 'verificacao', 'tentativa': 1, 'estado': None, 'erros': ['OutputParserException']}
{'evento': 'tool_start', 'rota': 'consultar_cep', 'ferramenta': 'consultar_cep', 'entrada': "{'cep': '01001000'}"}
{'evento': 'tool_end', 'resultado': '{"estado": "ok", "cep": "01001000", "cidade": "Cidade Simulada", "uf":', 'fontes': []}
{'evento': 'cep_ancorado', 'estado': 'ancorada', 'cep': '01001000'}
{'evento': 'verificacao', 'tentativa': 2, 'estado': 'cep', 'erros': []}
publicado: CEP 01001000: Cidade Simulada/SP (ViaCEP, modo simulado).
memória: [('human', 'A qual cidade corresponde o CEP 01001000?'), ('ai', 'CEP 01001000: Cidade Simulada/SP (ViaCEP, modo simulado).')]


/tmp/ipykernel_555114/2373929637.py:12: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  fluxo = criar_fluxo(cenario, memoria, llm=RunnableLambda(lambda prompt, **kw: next(roteiro)),


Leitura: o registro mostra a ferramenta chamada duas vezes, a primeira tentativa rejeitada com `OutputParserException`, a segunda aceita e o CEP preenchido a partir da observação. A proposta rejeitada não chega à memória: só são salvas a pergunta e a resposta publicada.

## O que o orquestrador traz: inventário de peças do LangChain no Pedro
Sem modelo. O inventário é obtido lendo os imports do código (`pedro.anatomia.inventario_langchain`), não é escrito à mão. Serve para argumentar a escolha com fatos: cada linha é uma peça que o projeto não precisou construir. A comparação executada com o LangGraph está em docs/rastros/r07_orquestradores_llm.json; aqui só o LangChain foi executado.

In [6]:
from pedro.anatomia import inventario_langchain
inventario = inventario_langchain()
for peca, dado in inventario.items():
    print(f"{peca:38} {', '.join(dado['nomes'])}")
    print(f"{'':38} usado em: {', '.join(dado['arquivos'])}")
print('peças:', len(inventario), '| nomes importados:', sum(len(d['nomes']) for d in inventario.values()))

agentes e executor                     AgentExecutor, AgentType, create_openai_tools_agent, initialize_agent
                                       usado em: agent.py, legacy.py, orquestradores.py
armazém vetorial (FAISS)               FAISS
                                       usado em: policy_demo.py, rag.py
cadeias clássicas (RouterChain)        ConversationChain, LLMRouterChain, MULTI_PROMPT_ROUTER_TEMPLATE, RouterChain, RouterOutputParser
                                       usado em: legacy.py
callbacks e rastros                    BaseCallbackHandler
                                       usado em: agent.py, laboratorio.py
composição (Runnable, LCEL, @chain)    ConfigurableFieldSpec, RunnableConfig, RunnableLambda, RunnableParallel, RunnablePassthrough, RunnableWithMessageHistory, chain
                                       usado em: agent.py, anatomia.py, chains.py, laboratorio.py, orquestradores.py, precos.py
divisão de documentos                  RecursiveCharacterTextSp